# Exploratory Data Analysis (EDA)
This notebook performs basic EDA on the first available CSV found in the repository `data/processed`, `data/raw`, or `data/interim`. It reports shape, types, missing values, basic summary statistics, and simple plots for numeric columns.

In [ ]:
# Imports
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Find repository root (look for .git) and search data folders
cwd = Path().resolve()
repo_root = cwd
while not (repo_root / '.git').exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
data_dirs = [repo_root / 'data' / 'processed', repo_root / 'data' / 'raw', repo_root / 'data' / 'interim']
csv_files = []
for d in data_dirs:
    if d.exists():
        csv_files.extend(sorted([str(d / f) for f in os.listdir(d) if f.lower().endswith('.csv')]))

print('Searched data dirs:', [str(p) for p in data_dirs])
print('Found CSVs:', csv_files)

# Prefer explicit file path if available (set by request)
preferred = repo_root / 'data' / 'processed' / 'nyc_annual_salary_employees_payBasis_perAnuum.csv'
if preferred.exists():
    data_path = str(preferred)
    print('Using preferred data path:', data_path)
else:
    if not csv_files:
        raise FileNotFoundError('No CSV files found in data/processed, data/raw, or data/interim')
    # fallback to the first discovered CSV
    data_path = csv_files[0]
    print('Preferred file not found; falling back to', data_path)

df = pd.read_csv(data_path, low_memory=False)

# Display first rows
df.head()

In [ ]:
# Basic information
print('Shape:', df.shape)
print('
Data types:')
print(df.dtypes)

print('
Missing values (top 20):')
print(df.isna().sum().sort_values(ascending=False).head(20))

print('
Numeric summary:')
display(df.describe(include=[np.number]).T)

In [ ]:
# Show top value counts for object columns (up to first 6 object columns)
obj_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in obj_cols[:6]:
    print('---', col, '---')
    print(df[col].value_counts(dropna=False).head(10))
    print()

In [ ]:
# Simple plots for numeric columns
num_df = df.select_dtypes(include=[np.number])
if not num_df.empty:
    num_cols = num_df.columns.tolist()
    print('Numeric columns:', num_cols[:10])
    # Histogram grid for first 8 numeric columns
    cols_to_plot = num_cols[:8]
    if cols_to_plot:
        fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16,8))
        axes = axes.flatten()
        for i, c in enumerate(cols_to_plot):
            sns.histplot(df[c].dropna(), ax=axes[i], kde=False)
            axes[i].set_title(c)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric columns found to plot.')

In [ ]:
# Save summary and outputs to eda_outputs (also creates plots)
out_dir = repo_root / 'notebooks' / 'exploratory' / 'eda_outputs'
plots_dir = out_dir / 'plots'
out_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)
summary_path = out_dir / 'summary_from_notebook.txt'
with open(summary_path, 'w', encoding='utf-8') as outf:
    outf.write(f'Loaded: {data_path}
')
    outf.write(f'Shape: {df.shape}

')
    outf.write('-- Dtypes --
')
    for col, dt in df.dtypes.items():
        outf.write(f'{col}: {dt}
')
    outf.write('
-- Top missing values (20) --
')
    outf.write(df.isna().sum().sort_values(ascending=False).head(20).to_string())
    outf.write('

-- Numeric summary (first 20 cols) --
')
    num = df.select_dtypes(include=[np.number])
    if not num.empty:
        outf.write(num.describe().T.head(20).to_string())
    else:
        outf.write('No numeric cols
')
    outf.write('

-- Top categories for up to 6 object cols --
')
    obj_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in obj_cols[:6]:
        outf.write(f'
[{col}] top values:
')
        outf.write(df[col].value_counts(dropna=False).head(10).to_string())
        outf.write('
')
# save head sample
df.head(200).to_csv(out_dir / 'head_sample.csv', index=False)
# save histograms for first up to 8 numeric columns
if not num.empty:
    for c in num.columns.tolist()[:8]:
        try:
            plt.figure(figsize=(6,4))
            sns.histplot(df[c].dropna(), kde=False)
            plt.title(c)
            plt.tight_layout()
            plt.savefig(plots_dir / f'hist_{c}.png')
            plt.close()
        except Exception as e:
            with open(summary_path,'a',encoding='utf-8') as outf:
                outf.write(f'Failed plotting {c}: {e}
')

print('Saved outputs to', out_dir)

## Next steps
- Run cell outputs and inspect unusual missingness or extreme values.
- Choose columns of interest and create targeted plots (boxplots, time series, category comparisons).
- Save a cleaned/filtered CSV to `data/interim` or `data/processed` as appropriate.